# 05c: Computational efficiency, DIWECT chunk sizes (Colab T4)

**Paper:** Section *Computational Efficiency*, last paragraph of *Cost at matched accuracy*: the DIWECT processes cells in chunks of a fixed number of (cell, vertex, height) triples, so its peak memory does not grow with the size of the complex.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = Path("/content").is_dir()
WORK = Path(os.environ.get("BENCH_DIR", "/content" if IN_COLAB else "runs/efficiency"))
INPUTS = WORK / "inputs.npz" if IN_COLAB else Path("data/efficiency/inputs.npz")
WORK.mkdir(parents=True, exist_ok=True)
SMOKE = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyect", "trimesh"], check=True)

import numpy as np
import torch

from diwect import DIWECT

GPU = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA = GPU.type == "cuda"
MEM_BUDGET = 4 * 2 ** 30
SLOW = 60.0
CASES = {
    2: ["img1024_gray_freudenthal", "img1024_bin_cubical"],
    3: ["seed_vox2_freudenthal", "seed_vox1_freudenthal", "seed_vox1_cubical"],
}
CHUNKS = {2: [2 ** k for k in range(26, 15, -1)], 3: [2 ** k for k in range(23, 13, -1)]}
HS = (65, 257)
if SMOKE:
    CASES = {2: ["img64_gray_freudenthal", "img64_bin_cubical"], 3: ["seed_mesh_step4", "seed_vox4_cubical"]}
    CHUNKS = {2: [2 ** k for k in (20, 16, 12)], 3: [2 ** k for k in (18, 14, 10)]}

data = np.load(INPUTS)
index = json.loads(bytes(data["index"]).decode())
OUT = open(WORK / "calibrate.jsonl", "w")


def emit(**rec):
    OUT.write(json.dumps(rec) + "\n")
    OUT.flush()
    print(json.dumps(rec), flush=True)


def load(name):
    return [(torch.as_tensor(data[f"{name}/{k}/x"]).to(GPU, torch.float64 if k == 0 else torch.long),
             torch.as_tensor(data[f"{name}/{k}/w"]).to(GPU, torch.float64))
            for k in range(len(index[name]["cells"]))]


def sync():
    if CUDA:
        torch.cuda.synchronize()

In [ ]:
chosen = {}
for dim, names in CASES.items():
    times = {}
    peaks = {}
    for name in names:
        cx = load(name)
        for h in HS:
            for chunk in CHUNKS[dim]:
                mod = DIWECT(h, normalize=True, chunk_size=chunk)
                try:
                    if CUDA:
                        torch.cuda.empty_cache()
                        base = torch.cuda.memory_allocated()
                        torch.cuda.reset_peak_memory_stats()
                    sync()
                    start = time.perf_counter()
                    mod(cx)
                    sync()
                    t = time.perf_counter() - start
                    peak = torch.cuda.max_memory_allocated() - base if CUDA else 0
                    if t < 5:
                        ts = []
                        for _ in range(3):
                            sync()
                            start = time.perf_counter()
                            mod(cx)
                            sync()
                            ts.append(time.perf_counter() - start)
                        t = float(np.median(ts))
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    emit(kind="calib", dim=dim, complex=name, cells=index[name]["total_cells"], h=h,
                         chunk_size=chunk, oom=True)
                    peaks[chunk] = float("inf")
                    continue
                times.setdefault((name, h), {})[chunk] = t
                peaks[chunk] = max(peaks.get(chunk, 0), peak)
                emit(kind="calib", dim=dim, complex=name, cells=index[name]["total_cells"], h=h,
                     chunk_size=chunk, seconds=t, peak_bytes=peak)
                if t > SLOW:
                    break
        del cx

    ok = []
    for chunk in sorted(CHUNKS[dim]):
        if peaks.get(chunk, float("inf")) > MEM_BUDGET:
            continue
        if all(chunk in ts and ts[chunk] <= 1.1 * min(ts.values()) for ts in times.values()):
            ok.append(chunk)
    if not ok:
        cands = [c for c in CHUNKS[dim] if peaks.get(c, float("inf")) <= MEM_BUDGET
                 and all(c in ts for ts in times.values())]
        ok = [min(cands, key=lambda c: sum(ts[c] for ts in times.values()))]
    chosen[dim] = ok[0]
    emit(kind="chosen", dim=dim, chunk_size=ok[0], peak_bytes=peaks[ok[0]])

json.dump(chosen, open(WORK / "chunk.json", "w"))
emit(kind="done", chosen=chosen)
OUT.close()